# 02. Three operational approaches, all free

There are several ways to obtain decision-model behaviour. This notebook compares three of them, all free or open:

1. A local open model served by Ollama or llama.cpp.
2. A free hosted route, used only if you do not have local hardware.
3. Training-free logit reading, where an open model scores our declared options directly.

We will load configurations, inspect endpoints, and compare clients on identical inputs.


## Step 1. Read the three configurations

Each YAML file describes a backend. We load them one at a time and print one field per cell, so it is clear where each value comes from.


In [ ]:
from decision_qa import load_config
local_ollama = load_config('configs/local_ollama.yaml')
print(local_ollama.backend)


In [ ]:
print(local_ollama.base_url)
print(local_ollama.endpoint)


In [ ]:
local_llamacpp = load_config('configs/local_llamacpp.yaml')
print(local_llamacpp.endpoint)


In [ ]:
hosted = load_config('configs/hosted.yaml')
print(hosted.backend)
print(hosted.model)


In [ ]:
assert local_ollama.endpoint.endswith('/v1/systemone')
assert hosted.endpoint.endswith('/v1/systemone')
print('all endpoints well formed')


## Step 2. The offline mock

The mock needs no server, key, or network. It is the right tool while learning the data structures and while running tests. We call it on a small state and two questions.


In [ ]:
from decision_qa import MockClient, NoulQuestion, ChoiceQuestion
state = {'context': 'The context says the treatment helped.', 'question': 'Did it help?'}
questions = [NoulQuestion(label='helped', statement='The treatment helped.', gold=True), ChoiceQuestion(label='verdict', options=['yes', 'no', 'unclear'], gold_index=0)]


In [ ]:
mock = MockClient(seed=0)
mock_results = mock.decision(state, questions)
print([r.question_label for r in mock_results])


In [ ]:
for r in mock_results:
    print(r.question_label, round(max(r.probabilities.values()), 3))


## Step 3. A live client, only if a server is running

We instantiate a real client but we do not assume a server exists. The `try` block attempts the call and reports a friendly message if no local model is running. This keeps the notebook executable on a fresh machine.


In [ ]:
from decision_qa import SystemOneClient
live_local = SystemOneClient(local_ollama)
print(live_local.config.endpoint)


In [ ]:
try:
    live_local.decision(state, questions)
except Exception as exc:
    print('local model not running:', type(exc).__name__)
else:
    print('local model answered')


## Step 4. Training-free logit reading

Open methods such as LitJev, SemIf, and AnyJev turn any open model into a decision model without fine-tuning. The trick is to read the probability of the label tokens directly from one forward pass. Our `question_labels` helper shows exactly which labels would be read for a given question.


In [ ]:
from decision_qa import question_labels
readout = question_labels(questions[1])
print(readout)


In [ ]:
assert readout == ['yes', 'no', 'unclear']
print('these are the option tokens to score')


## Step 5. Compare latency on identical inputs

We time each available client on the same state and questions. On this machine only the mock is available, so the comparison is a template you can complete after starting a free local model.


In [ ]:
import time
def timed(client, label):
    start = time.perf_counter()
    client.decision(state, questions)
    elapsed = (time.perf_counter() - start) * 1000
    print(label, round(elapsed, 1), 'ms')


In [ ]:
timed(mock, 'mock')


## Step 6. Choosing an approach

Use a local open model for privacy, offline work, and full control. Use a free hosted route only when you lack local hardware and the free tier is enough. Use logit reading for fast prototypes where no training is acceptable.

Open implementations worth reading: LitJev, SemIf, AnyJev, and the llama.cpp decision support documented in the curated awesome-decision-models list.
